# 面试题：token、word、character 分别是什么，工程上为什么不能混用？

## 面试回答主线

`character` 是 Unicode 字符层面的单位，但用户看到的一个“字形”可能由多个字符组成，UTF-8 中一个字符也可能占多个字节。`word` 是带语义的语言单位，英文常由空格近似分隔，中文却需要分词或词典，因此它不是稳定的存储单位。`token` 是某个 tokenizer 按固定词表和规则产生的模型输入单位，它可能是一个词、半个词、一个汉字、标点，甚至一个 UTF-8 字节。模型只接收 token id，所以同一句话在不同 tokenizer 下长度、费用和截断位置都可能不同。工程上必须同时保存 tokenizer 版本与规范化策略，否则训练、评测和线上服务会发生静默漂移。下面用客服工单展示字符、词、token、字节和业务字段之间的真实差异。

## 真实案例：多语言客服工单

数据是手写的脱敏教学样本，字段结构来自常见客服系统：工单号、渠道和原始文本。它覆盖中文、英文、组合字符、emoji、URL 与全角字符；规模很小，只用于解释机制，不能外推线上 token 成本。

In [1]:
import re  # 导入正则表达式以完成预切分和字段识别。
import unicodedata  # 导入 Unicode 工具以观察字符名称和执行规范化。
from collections import Counter  # 导入计数器以汇总词片频率。
records = [  # 构造具有真实业务字段的脱敏客服工单。
    {"ticket_id": "TK-1001", "channel": "app", "text": "订单A98372退款已经到账了吗？"},  # 中文文本同时包含订单号和问号。
    {"ticket_id": "TK-1002", "channel": "web", "text": "Bluetooth headphones won't connect."},  # 英文文本用空格近似分词。
    {"ticket_id": "TK-1003", "channel": "email", "text": "蓝牙耳机🎧无法连接 iPhone15"},  # 混合中文、emoji 与英文数字。
    {"ticket_id": "TK-1004", "channel": "app", "text": "优惠券 SAVE-20% 显示已失效"},  # 促销码包含连字符和百分号。
    {"ticket_id": "TK-1005", "channel": "web", "text": "咖啡e\u0301已洒，请看 https://ex.am/p/7"},  # 组合重音字符与 URL 会暴露字符边界问题。
    {"ticket_id": "TK-1006", "channel": "api", "text": "签名字段=ＡＢＣ-007，请勿改写"},  # 全角机器标识符会暴露规范化风险。
]  # 结束教学工单列表。
print("教学实验输入：共", len(records), "条工单")  # 输出样本规模以明确实验边界。
for item in records:  # 逐条展示可读的原始记录而不是只打印变量名。
    print(f"{item['ticket_id']} | {item['channel']:<5} | {item['text']}")  # 输出工单号、渠道和文本。

教学实验输入：共 6 条工单
TK-1001 | app   | 订单A98372退款已经到账了吗？
TK-1002 | web   | Bluetooth headphones won't connect.
TK-1003 | email | 蓝牙耳机🎧无法连接 iPhone15
TK-1004 | app   | 优惠券 SAVE-20% 显示已失效
TK-1005 | web   | 咖啡é已洒，请看 https://ex.am/p/7
TK-1006 | api   | 签名字段=ＡＢＣ-007，请勿改写


## Character 与字节：先看用户肉眼看不到的边界

Python 的 `len(str)` 统计 Unicode code point，不是 UTF-8 字节数，也不一定等于屏幕上的字形数。样本中的 `e + 组合重音` 看起来像一个 `é`，却由两个 character 构成；emoji 还可能带变体选择符。

In [2]:
def unicode_rows(text):  # 定义字符检查函数以暴露码点、字节和名称。
    rows = []  # 创建列表保存每个字符的可解释信息。
    for index, character in enumerate(text):  # 按 Python 的 Unicode 字符边界遍历文本。
        code_point = f"U+{ord(character):04X}"  # 把字符转换为标准 Unicode 码点表示。
        byte_count = len(character.encode("utf-8"))  # 计算当前字符在 UTF-8 下占用的字节数。
        name = unicodedata.name(character, "未命名字符")  # 查询字符名称以识别组合符和变体符。
        rows.append((index, repr(character), code_point, byte_count, name))  # 保存当前字符的完整观察结果。
    return rows  # 返回字符级明细供调用方展示。
focus_text = "e\u0301 与 🎧️"  # 选择包含组合重音和 emoji 变体符的最小反例。
print("索引 | 字符 | 码点 | UTF-8字节 | Unicode 名称")  # 输出字符表的列标题。
for row in unicode_rows(focus_text):  # 遍历并展示每个字符的底层表示。
    print(f"{row[0]:>4} | {row[1]:^6} | {row[2]:<8} | {row[3]:>10} | {row[4]}")  # 输出一行字符明细。
print("汇总：Python字符数=", len(focus_text), "，UTF-8字节数=", len(focus_text.encode("utf-8")))  # 对比字符数与传输字节数。

索引 | 字符 | 码点 | UTF-8字节 | Unicode 名称
   0 |  'e'   | U+0065   |          1 | LATIN SMALL LETTER E
   1 |  '́'   | U+0301   |          2 | COMBINING ACUTE ACCENT
   2 |  ' '   | U+0020   |          1 | SPACE
   3 |  '与'   | U+4E0E   |          3 | CJK UNIFIED IDEOGRAPH-4E0E
   4 |  ' '   | U+0020   |          1 | SPACE
   5 |  '🎧'   | U+1F3A7  |          4 | HEADPHONE
   6 |  '️'   | U+FE0F   |          3 | VARIATION SELECTOR-16
汇总：Python字符数= 7 ，UTF-8字节数= 15


## Baseline：空格切 word、逐字符当 token

最朴素方案常把空格片段叫作 word，把非空字符直接当 token。它在英文上勉强可读，但整句中文会被算成一个 word；逐字符 token 又无法复用“蓝牙耳机”等高频词片。先把这个错误基线量化出来。

In [3]:
def whitespace_words(text):  # 定义只依赖空白符的朴素分词基线。
    return re.findall(r"\S+", text)  # 返回连续非空白片段并保留其原始形式。
def character_tokens(text):  # 定义逐 Unicode 字符切分的朴素 tokenizer。
    return [character for character in text if not character.isspace()]  # 忽略空白并把每个字符视为一个 token。
print("工单     空格word数  character-token数  UTF-8字节数  空格切分结果")  # 输出基线对比表标题。
for item in records:  # 在同一批工单上计算三种长度口径。
    words = whitespace_words(item["text"])  # 使用空格切分得到所谓 word。
    characters = character_tokens(item["text"])  # 使用逐字符基线得到 token 列表。
    byte_count = len(item["text"].encode("utf-8"))  # 计算网络和磁盘更关心的 UTF-8 字节数。
    print(f"{item['ticket_id']:<8} {len(words):>10} {len(characters):>18} {byte_count:>12}  {words}")  # 输出逐工单对照结果。

工单     空格word数  character-token数  UTF-8字节数  空格切分结果
TK-1001           1                 17           39  ['订单A98372退款已经到账了吗？']
TK-1002           4                 32           35  ['Bluetooth', 'headphones', "won't", 'connect.']
TK-1003           2                 17           37  ['蓝牙耳机🎧无法连接', 'iPhone15']
TK-1004           3                 16           34  ['优惠券', 'SAVE-20%', '显示已失效']
TK-1005           2                 26           42  ['咖啡é已洒，请看', 'https://ex.am/p/7']
TK-1006           1                 17           41  ['签名字段=ＡＢＣ-007，请勿改写']


## 核心实现：领域 word 与模型 token 分成两条管线

下面先用最长匹配词典产生便于搜索、统计和展示的 `word`，再用一个冻结的小词片表产生模型 `token`。词片表命中时一个 token 可覆盖多个 character；未命中字符退化为 UTF-8 byte token，从而不丢失任何输入。真实系统的词表来自大语料训练，这里手写小词表只是为了把决策路径完整展示出来。

In [4]:
domain_words = ["Bluetooth", "headphones", "蓝牙耳机", "无法连接", "订单", "退款", "已经到账", "优惠券", "显示", "已失效", "签名字段", "请勿改写", "咖啡", "已洒", "请看"]  # 定义客服领域的可解释词典。
domain_words = sorted(domain_words, key=lambda word: (-len(word), word))  # 按长度降序保证最长词优先且结果稳定。
def segment_words(text):  # 定义面向业务语义的最长匹配 word 分词器。
    words = []  # 创建列表保存识别出的业务词和回退片段。
    index = 0  # 从原始文本首字符开始扫描。
    while index < len(text):  # 持续扫描直到消费完整段文本。
        if text[index].isspace():  # 空白只作为边界而不产生业务词。
            index += 1  # 向后移动一个字符跳过当前空白。
            continue  # 回到循环顶部处理下一个位置。
        matched = next((word for word in domain_words if text.startswith(word, index)), None)  # 在当前位置寻找最长领域词。
        if matched is not None:  # 找到领域词时优先保留完整语义单位。
            words.append(matched)  # 把命中的领域词加入结果。
            index += len(matched)  # 一次跨过领域词覆盖的全部字符。
            continue  # 回到循环顶部继续扫描剩余文本。
        ascii_match = re.match(r"[A-Za-z0-9]+(?:[-_.:/][A-Za-z0-9]+)*", text[index:])  # 尝试识别订单号、URL 片段或英文串。
        if ascii_match is not None:  # 连续 ASCII 业务字段应作为一个 word。
            words.append(ascii_match.group(0))  # 保存完整的 ASCII 业务片段。
            index += len(ascii_match.group(0))  # 跳过刚刚消费的 ASCII 片段。
            continue  # 回到循环顶部继续扫描。
        words.append(text[index])  # 对未命中的汉字或标点执行单字符回退。
        index += 1  # 向后移动一个字符保证循环推进。
    return words  # 返回业务语义层面的 word 序列。
piece_vocabulary = ["bluetooth", "headphones", "蓝牙", "耳机", "无法", "连接", "订单", "退款", "已经", "到账", "优惠", "券", "显示", "失效", "签名", "字段", "请勿", "改写", "咖啡", "已洒", "请看", "https://", ".", "-", "%", "=", "？"]  # 定义演示用冻结模型词片表。
piece_vocabulary = sorted(piece_vocabulary, key=lambda piece: (-len(piece), piece))  # 按长度降序实现确定性的最长词片匹配。
def tokenize(text, normalize=True):  # 定义包含规范化、词片命中和字节回退的 tokenizer。
    prepared = unicodedata.normalize("NFKC", text).casefold() if normalize else text  # 根据策略生成真正进入 tokenizer 的文本视图。
    tokens = []  # 创建列表保存最终 token 字符串。
    index = 0  # 从规范化文本首字符开始扫描。
    while index < len(prepared):  # 持续扫描直到完整消费文本。
        if prepared[index].isspace():  # 本教学 tokenizer 不为普通空白分配 token。
            index += 1  # 跳过当前空白字符。
            continue  # 回到循环顶部处理下一位置。
        matched = next((piece for piece in piece_vocabulary if prepared.startswith(piece, index)), None)  # 查找当前位置可用的最长词片。
        if matched is not None:  # 词表命中时一个 token 可以覆盖多个字符。
            tokens.append(matched)  # 保存命中的词片文本。
            index += len(matched)  # 跨过该词片覆盖的字符范围。
            continue  # 回到循环顶部继续处理剩余文本。
        raw_bytes = prepared[index].encode("utf-8")  # 把未登录字符编码为可逆的 UTF-8 字节。
        tokens.extend([f"<0x{value:02X}>" for value in raw_bytes])  # 为每个字节创建不会产生未知词的回退 token。
        index += 1  # 当前 Unicode 字符已经通过字节回退完整消费。
    return prepared, tokens  # 同时返回规范化文本与 token 以便审计。
all_tokens = []  # 收集样本中实际出现的 token 以创建演示 token id。
for item in records:  # 遍历全部教学工单构造固定映射。
    all_tokens.extend(tokenize(item["text"])[1])  # 追加当前文本产生的所有 token。
token_to_id = {token: index + 4 for index, token in enumerate(sorted(set(all_tokens)))}  # 稳定分配普通 token id 并预留特殊编号。
token_to_id.update({"[PAD]": 0, "[UNK]": 1, "[BOS]": 2, "[EOS]": 3})  # 加入模型批处理常用的四个特殊 token。
focus = records[2]["text"]  # 选择混合语言工单展示完整处理链路。
prepared_focus, focus_tokens = tokenize(focus)  # 对焦点工单执行 tokenizer。
focus_ids = [token_to_id[token] for token in focus_tokens]  # 把可读 token 映射为模型真正接收的整数 id。
print("原文：", focus)  # 输出未经处理的业务文本。
print("word：", segment_words(focus))  # 输出面向业务理解的 word 序列。
print("规范化视图：", prepared_focus)  # 输出 tokenizer 实际看到的文本。
print("token：", focus_tokens)  # 输出词片和字节回退组成的 token 序列。
print("token id：", focus_ids)  # 输出可直接送入 embedding 层的整数序列。

原文： 蓝牙耳机🎧无法连接 iPhone15
word： ['蓝牙耳机', '🎧', '无法连接', 'iPhone15']
规范化视图： 蓝牙耳机🎧无法连接 iphone15
token： ['蓝牙', '耳机', '<0xF0>', '<0x9F>', '<0x8E>', '<0xA7>', '无法', '连接', '<0x69>', '<0x70>', '<0x68>', '<0x6F>', '<0x6E>', '<0x65>', '<0x31>', '<0x35>']
token id： [65, 64, 47, 38, 35, 39, 61, 69, 24, 28, 23, 27, 26, 22, 11, 14]


## 结果表：同一文本的四种“长度”会回答不同问题

`word` 数适合搜索召回和业务展示，`character` 数接近文本界面限制，UTF-8 字节数用于存储与传输，`token` 数决定模型上下文占用和常见计费口径。它们不能互相替代。

In [5]:
print("工单     word数  character数  token数  UTF-8字节  token相对逐字符")  # 输出统一口径的结果表标题。
token_counts = {}  # 保存每条工单的 token 数供后续回归测试和成本估算。
for item in records:  # 在相同输入上比较全部长度口径。
    word_count = len(segment_words(item["text"]))  # 计算领域分词后的 word 数。
    character_count = len(character_tokens(item["text"]))  # 计算去空白后的 Unicode 字符数。
    current_tokens = tokenize(item["text"])[1]  # 执行演示 tokenizer 得到最终词片。
    token_count = len(current_tokens)  # 统计模型上下文真正消耗的 token 数。
    token_counts[item["ticket_id"]] = token_count  # 按工单号保存 token 长度。
    byte_count = len(item["text"].encode("utf-8"))  # 统计原始 UTF-8 传输字节数。
    ratio = token_count / character_count  # 计算 token 与逐字符基线的长度比。
    print(f"{item['ticket_id']:<8} {word_count:>6} {character_count:>12} {token_count:>8} {byte_count:>11} {ratio:>16.2f}")  # 输出逐样本对照行。
price_per_million = 2.0  # 假设输入价格为每百万 token 两元以演示成本换算。
estimated_cost = sum(token_counts.values()) / 1_000_000 * price_per_million  # 按模型 token 而不是字符或字节估算费用。
print(f"这 {len(records)} 条样本共 {sum(token_counts.values())} token，教学价格下约 ¥{estimated_cost:.6f}")  # 输出可复核的成本估算。

工单     word数  character数  token数  UTF-8字节  token相对逐字符
TK-1001       7           17       17          39             1.00
TK-1002       7           32       15          35             0.47
TK-1003       4           17       16          37             0.94
TK-1004       5           16       15          34             0.94
TK-1005      11           26       16          42             0.62
TK-1006       9           17       13          41             0.76
这 6 条样本共 92 token，教学价格下约 ¥0.000184


## 结果解读

中文工单在空格基线下几乎总是一个 word，说明“按空格数词”不可用。高频词片能把“蓝牙”“耳机”“订单”“退款”压成少量 token；未覆盖的 emoji 和生僻字符会展开成多个 byte token，因此 token 数甚至可能大于 character 数。token id 只在“词表 + 规范化规则 + 特殊符号编号”版本固定时有意义，不能把另一个 tokenizer 的 id 直接复用。

## 失败案例：对整条业务记录做 NFKC 会悄悄改写机器标识符

文本规范化有助于召回和词表覆盖，但签名、哈希、优惠码和订单号属于机器字段，任何字符变化都可能导致验签失败。下面先复现错误，再通过字段级策略修复。

In [6]:
raw_payload = {"message": "请核对以下签名字段", "signature": "ＡＢＣ-007"}  # 构造同时包含自然语言和全角签名的业务载荷。
naive_payload = {key: unicodedata.normalize("NFKC", value).casefold() for key, value in raw_payload.items()}  # 错误地对所有字符串字段统一规范化。
def prepare_payload(payload):  # 定义按字段语义选择规范化策略的修复方案。
    prepared = dict(payload)  # 复制原始载荷以避免修改调用方数据。
    prepared["message"] = unicodedata.normalize("NFKC", payload["message"]).casefold()  # 只规范化供模型理解的自然语言字段。
    prepared["signature"] = payload["signature"]  # 对需要精确匹配的机器字段保留原始码点。
    return prepared  # 返回可同时服务模型与验签逻辑的字段级结果。
fixed_payload = prepare_payload(raw_payload)  # 对同一载荷应用字段级修复策略。
print("原始签名：", raw_payload["signature"], [f"U+{ord(ch):04X}" for ch in raw_payload["signature"][:3]])  # 输出原签名和前三个全角码点。
print("错误规范化：", naive_payload["signature"], "，是否仍可精确验签：", naive_payload["signature"] == raw_payload["signature"])  # 展示全局规范化导致的静默变化。
print("字段级修复：", fixed_payload["signature"], "，是否仍可精确验签：", fixed_payload["signature"] == raw_payload["signature"])  # 展示修复后机器字段保持不变。

原始签名： ＡＢＣ-007 ['U+FF21', 'U+FF22', 'U+FF23']
错误规范化： abc-007 ，是否仍可精确验签： False
字段级修复： ＡＢＣ-007 ，是否仍可精确验签： True


## 生产差距与落地清单

教学词表只有几十个词片，线上应使用经过版本管理的大语料 tokenizer，并固定 Unicode 版本、规范化策略、特殊 token、padding 方向和截断规则。服务应监控每语言 token/character 比、byte fallback 比例、截断率与 tokenizer 版本分布；升级词表时必须回放真实流量并比较 token id、长度、成本和关键字段偏移。若要做实体高亮，还需保存从规范化文本到原文的 offset 映射，而不能直接拿规范化后的索引切原字符串。

## 最小回归测试

断言只保护本实验最关键的不变量；学习证据已经由上面的输入、过程表、失败复现和修复对照给出。

In [7]:
assert len(segment_words("蓝牙耳机无法连接")) == 2  # 验证领域最长匹配能保留两个完整语义词。
assert tokenize("蓝牙耳机")[1] == ["蓝牙", "耳机"]  # 验证模型 token 可以跨越多个 character。
assert all(token != "[UNK]" for token in tokenize("🎧")[1])  # 验证 UTF-8 字节回退不会产生未知词。
assert fixed_payload["signature"] == raw_payload["signature"]  # 验证字段级策略不会改写精确机器标识符。
assert naive_payload["signature"] != raw_payload["signature"]  # 固化能够暴露全局规范化风险的反例。
print("最小回归测试通过：词边界、词片覆盖、字节回退和字段级规范化均符合预期。")  # 输出测试结论方便读者确认完整运行。

最小回归测试通过：词边界、词片覆盖、字节回退和字段级规范化均符合预期。
